# Liberia FY2026 Math & Science — Bong + Nimba aggregation

This Python 3.10+ notebook uses **openpyxl**, not `artifact_tool` or `artifact_tool_v2`. It aggregates the indicator quarterly summary columns from `STEMBong.xlsx` and `STEMNimba.xlsx` by **indicator, sex, and age** into a copy of `Liberia_PSR_Math&Science_FY26.xlsx`.

The notebook preserves the reference workbook's FY targets and formatting. It adds source-detail and reference-comparison worksheets. It **does not** count activity rows as separate totals. Blank/unverified source cells are not silently treated as zeros. By default, each value uses its latest nonblank quarterly snapshot (Q1–Q4), not the sum of quarters.

**Setup:** Place this notebook and all three input `.xlsx` files in the same folder, then run the cells in order. If the first cell reports `ModuleNotFoundError`, uncomment and run `%pip install openpyxl` in that cell and restart the kernel.

In [3]:
# Uncomment the next line if openpyxl is not installed in this Jupyter kernel:
# %pip install openpyxl

from collections import defaultdict
from pathlib import Path
import re
from copy import copy
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

In [4]:
# By default, look for input files in the notebook's current working directory.
INPUT_DIR = Path('.')
BONG_FILE = INPUT_DIR / 'data/redataatindividualpcvlevel/STEMBong.xlsx'
NIMBA_FILE = INPUT_DIR / 'data/redataatindividualpcvlevel/STEMNimba.xlsx'
REFERENCE_FILE = INPUT_DIR / 'data/projectstatusreports/Liberia_PSR_Math&Science_FY26.xlsx'
OUTPUT_FILE = INPUT_DIR / 'Liberia_PSR_Bong_Nimba_FY26_Aggregated.xlsx'
SOURCE_FILES = [BONG_FILE, NIMBA_FILE]
QUARTER_STRATEGY = 'latest_nonblank'

for required in [*SOURCE_FILES, REFERENCE_FILE]:
    if not required.is_file():
        raise FileNotFoundError(f'Missing input file: {required.resolve()}')

print('Input files found:', *(p.name for p in [*SOURCE_FILES, REFERENCE_FILE]), sep='\n  ')

Input files found:
  STEMBong.xlsx
  STEMNimba.xlsx
  Liberia_PSR_Math&Science_FY26.xlsx


In [5]:
def numeric(value):
    """Return numeric counts; distinguish empty/unverified cells from actual zeros."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return value
    if isinstance(value, str):
        cleaned = value.strip().replace(',', '')
        if cleaned:
            try:
                return float(cleaned) if '.' in cleaned else int(cleaned)
            except ValueError:
                pass
    return None


def quarter_snapshot(row, quarter_indices):
    found = []
    for q in range(1, 5):
        col = quarter_indices.get(q)
        if col is not None:
            amount = numeric(row[col])
            if amount is not None:
                found.append((q, amount))
    if not found:
        return None, ''
    if QUARTER_STRATEGY != 'latest_nonblank':
        raise ValueError(f'Unsupported QUARTER_STRATEGY: {QUARTER_STRATEGY}')
    q, amount = found[-1]
    return amount, f'Q{q}'


def read_county_report(path):
    """Collect disaggregation rows from each county indicator sheet."""
    wb = load_workbook(path, data_only=True, read_only=True)
    details = []
    reporting = defaultdict(set)
    try:
        for sheet in wb.worksheets:
            # County source templates use A:U, with headers on rows 3, 4 and 7.
            matrix = [list(row) for row in sheet.iter_rows(
                min_row=1, max_row=45, min_col=1, max_col=21,
                values_only=True
            )]
            indicator = sheet.title.strip()
            indicator_label = str(matrix[3][1] or '')
            if indicator not in indicator_label:
                raise ValueError(
                    f'Indicator mismatch in {path.name} / {indicator}: {indicator_label}'
                )
            volunteer = str(matrix[2][1] or '').split(' (')[0].strip()
            reporting[indicator].add((path.stem, volunteer))

            reached_cols, achieved_cols = {}, {}
            for j, heading in enumerate(matrix[6]):
                match = re.fullmatch(
                    r'(Reached|Achieved) Q([1-4])',
                    str(heading or '').strip(), flags=re.I
                )
                if match:
                    group = reached_cols if match.group(1).lower() == 'reached' else achieved_cols
                    group[int(match.group(2))] = j
            if not reached_cols:
                raise ValueError(f'Quarterly reached columns missing: {path.name} / {indicator}')

            sex = ''
            for excel_row in range(8, len(matrix) + 1):
                row = matrix[excel_row - 1]
                if str(row[0] or '').strip().lower() == 'key:':
                    break
                if str(row[0] or '').strip().startswith('Sex:'):
                    sex = str(row[0]).strip()
                age = str(row[1] or '').strip()
                if not sex or (age and not age.startswith('Age:')):
                    continue
                if not age and not str(row[0] or '').startswith('Sex:'):
                    continue
                reached, reached_q = quarter_snapshot(row, reached_cols)
                achieved, achieved_q = quarter_snapshot(row, achieved_cols)
                details.append({
                    'Source': path.name,
                    'Volunteer': volunteer,
                    'Indicator': indicator,
                    'Sex': sex,
                    'Age': age,
                    'Reached': reached,
                    'Achieved': achieved,
                    'Reached quarter': reached_q,
                    'Achieved quarter': achieved_q,
                    'Verification': str(row[12 if achieved_cols else 8] or ''),
                    'Excel source row': excel_row,
                })
    finally:
        wb.close()
    return details, reporting


def write_rows(ws, first_row, first_col, rows):
    for row_i, values in enumerate(rows, start=first_row):
        for col_i, value in enumerate(values, start=first_col):
            ws.cell(row=row_i, column=col_i).value = value


def style_new_sheet(ws, right_col, widths):
    """Keep added reporting sheets readable, without changing the PSR template."""
    for row_num, bg, font_size, height in (
        (1, '103C51', 14, 32),
        (4, '164A62', 11, 32),
    ):
        for col in range(1, right_col + 1):
            cell = ws.cell(row_num, col)
            cell.fill = PatternFill('solid', fgColor=bg)
            cell.font = Font(bold=True, color='FFFFFF', size=font_size)
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        ws.row_dimensions[row_num].height = height
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    ws.freeze_panes = 'A5'
    ws.sheet_view.showGridLines = False

In [6]:
all_details = []
reporters = defaultdict(set)
for path in SOURCE_FILES:
    rows, county_reporters = read_county_report(path)
    all_details.extend(rows)
    for indicator, who in county_reporters.items():
        reporters[indicator].update(who)

aggregation = defaultdict(lambda: {
    'Reached': 0, 'Achieved': 0, 'has_reached': False, 'has_achieved': False
})
for item in all_details:
    key = (item['Indicator'], item['Sex'], item['Age'])
    target = aggregation[key]
    for measure in ('Reached', 'Achieved'):
        if item[measure] is not None:
            target[measure] += item[measure]
            target['has_' + measure.lower()] = True

unique_volunteers = {(r['Source'], r['Volunteer']) for r in all_details}
print(f'Indicator sheets: {sum(map(len, reporters.values()))}')
print(f'Disaggregation source rows: {len(all_details)}')
print(f'County/volunteer reports: {len(unique_volunteers)}')
print(f'Distinct indicator codes: {len(reporters)}')

Indicator sheets: 14
Disaggregation source rows: 60
County/volunteer reports: 2
Distinct indicator codes: 9


In [7]:
# Load original PSR without losing styles; J and M targets stay unchanged.
report_workbook = load_workbook(REFERENCE_FILE)
report_sheet = report_workbook.worksheets[0]
reference_rows = [
    [report_sheet.cell(row=r, column=c).value for c in range(1, 14)]
    for r in range(1, 193)
]

indicator_blocks = []
current_code, current_category, start_row = None, None, None
for excel_row in range(4, 193):
    values = reference_rows[excel_row - 1]
    if str(values[1] or '').strip() in ('Output', 'Outcome'):
        current_category = str(values[1]).strip()
    match = re.match(r'^(ED_MS_\d+(?:_[A-Z])?):', str(values[2] or ''))
    if match:
        current_code, start_row = match.group(1), excel_row
    if str(values[3] or '').strip() == 'Total':
        if not current_code or start_row is None:
            raise ValueError(f'Unknown indicator on PSR total row {excel_row}')
        indicator_blocks.append({
            'code': current_code, 'category': current_category,
            'first': start_row, 'last': excel_row - 1, 'total': excel_row,
            'reference_reached': numeric(values[6]),
            'reference_achieved': numeric(values[7]),
        })
if not indicator_blocks:
    raise ValueError('No PSR indicator blocks found; check the reference file.')

# Clear and replace existing Bong+Nimba count fields without modifying FY targets.
for r in range(4, 193):
    for c in (7, 8, 9, 12):
        report_sheet.cell(r, c).value = None

for block in indicator_blocks:
    code = block['code']
    for excel_row in range(block['first'], block['last'] + 1):
        ref = reference_rows[excel_row - 1]
        sex, age = str(ref[3] or '').strip(), str(ref[4] or '').strip()
        counts = aggregation.get((code, sex, age))
        if counts:
            if counts['has_reached']:
                report_sheet.cell(excel_row, 7).value = counts['Reached']
            if block['category'] == 'Outcome' and counts['has_achieved']:
                report_sheet.cell(excel_row, 8).value = counts['Achieved']

    a, b, total = block['first'], block['last'], block['total']
    report_sheet.cell(total, 7).value = f'=IF(COUNT(G{a}:G{b})=0,"",SUM(G{a}:G{b}))'
    report_sheet.cell(total, 12).value = len(reporters[code])
    if block['category'] == 'Outcome':
        report_sheet.cell(total, 8).value = f'=IF(COUNT(H{a}:H{b})=0,"",SUM(H{a}:H{b}))'
        for r in range(a, b + 1):
            report_sheet.cell(r, 9).value = f'=IF(OR(G{r}="",H{r}=""),"",IFERROR(H{r}/G{r},""))'
        report_sheet.cell(total, 9).value = (
            f'=IF(OR(G{total}="",H{total}=""),"",IFERROR(H{total}/G{total},""))'
        )
        numerator = f'H{total}'
    else:
        report_sheet.cell(total, 8).value = 'N/A'
        report_sheet.cell(total, 9).value = 'N/A'
        numerator = f'G{total}'
    report_sheet.cell(total, 11).value = (
        f'=IF(J{total}>0,IF({numerator}="",0,{numerator}/J{total}),"")'
    )

report_sheet['A1'] = 'Project Status Report: Math & Science FY2026 — Bong + Nimba'
report_sheet['A2'] = (
    f'{len(unique_volunteers)} volunteer reports combined | FY targets retained from national PSR'
)
for row in report_sheet.iter_rows(min_row=4, max_row=192, min_col=7, max_col=9):
    for cell in row:
        cell.number_format = '0.0%' if cell.column == 9 else '#,##0'
for row in report_sheet.iter_rows(min_row=4, max_row=192, min_col=11, max_col=11):
    row[0].number_format = '0.0%'

In [8]:
audit_sheet = report_workbook.create_sheet('Source Breakdown')
audit_headers = [
    'County workbook', 'Volunteer', 'Indicator', 'Sex', 'Age',
    'Reached', 'Achieved', 'Reached quarter', 'Achieved quarter',
    'Staff verification', 'Source row',
]
audit_sheet['A1'] = 'SOURCE BREAKDOWN | Bong + Nimba FY2026'
audit_sheet['A2'] = (
    'Uses quarterly summary cells, not activity totals; blanks are not automatically interpreted as zero.'
)
write_rows(audit_sheet, 4, 1, [audit_headers])
audit_rows = [[
    r['Source'], r['Volunteer'], r['Indicator'], r['Sex'], r['Age'],
    r['Reached'], r['Achieved'], r['Reached quarter'], r['Achieved quarter'],
    r['Verification'], r['Excel source row'],
] for r in all_details]
write_rows(audit_sheet, 5, 1, audit_rows)
style_new_sheet(audit_sheet, 11, {
    'A':19, 'B':29, 'C':17, 'D':18, 'E':18, 'F':15, 'G':15,
    'H':19, 'I':19, 'J':19, 'K':13,
})
for row in audit_sheet.iter_rows(min_row=5, max_row=max(5, len(audit_rows) + 4),
                                 min_col=6, max_col=7):
    for cell in row:
        cell.number_format = '#,##0'
if audit_rows:
    audit_sheet.auto_filter.ref = f'A4:K{len(audit_rows) + 4}'

In [9]:
reconcile = report_workbook.create_sheet('Reference Comparison')
reconcile['A1'] = 'TWO-REPORT AGGREGATION vs. SUPPLIED NATIONAL PSR'
reconcile['A2'] = (
    'Differences are informational: the two county workbooks may cover different people, '
    'periods or reporting rules.'
)
comparison_headers = [
    'Indicator', 'Reporting volunteers', 'Bong+Nimba reached',
    'Reference reached', 'Reached difference', 'Bong+Nimba achieved',
    'Reference achieved', 'Achieved difference', 'FY target (national)',
    'Indicator type',
]
write_rows(reconcile, 4, 1, [comparison_headers])
comparison_rows = []
for block in indicator_blocks:
    code = block['code']
    relevant = [v for (indicator, sex, age), v in aggregation.items() if indicator == code]
    has_reached = any(v['has_reached'] for v in relevant)
    has_achieved = any(v['has_achieved'] for v in relevant)
    reached = sum(v['Reached'] for v in relevant if v['has_reached']) if has_reached else None
    achieved = (
        sum(v['Achieved'] for v in relevant if v['has_achieved'])
        if has_achieved and block['category'] == 'Outcome' else None
    )
    ref_r, ref_a = block['reference_reached'], block['reference_achieved']
    fy_target = reference_rows[block['total'] - 1][9]
    comparison_rows.append([
        code, len(reporters[code]), reached, ref_r,
        (reached - ref_r) if reached is not None and ref_r is not None else None,
        achieved, ref_a,
        (achieved - ref_a) if achieved is not None and ref_a is not None else None,
        fy_target, block['category'],
    ])
write_rows(reconcile, 5, 1, comparison_rows)
style_new_sheet(reconcile, 10, {
    'A': 18, 'B': 23, 'C': 23, 'D': 23, 'E': 23,
    'F': 23, 'G': 23, 'H': 23, 'I': 23, 'J': 18,
})
if comparison_rows:
    reconcile.auto_filter.ref = f'A4:J{len(comparison_rows) + 4}'
for row in reconcile.iter_rows(min_row=5, max_row=max(5, len(comparison_rows) + 4),
                               min_col=2, max_col=9):
    for cell in row:
        cell.number_format = '#,##0'

In [10]:
# Check source-level sums before writing.
assert all(b['code'] for b in indicator_blocks)
assert len(all_details) > 0
missing_indicators = sorted(set(reporters) - {b['code'] for b in indicator_blocks})
if missing_indicators:
    print('WARNING: County indicators absent from national PSR:', ', '.join(missing_indicators))
for row in comparison_rows:
    code, reporting_count, reached, _, _, achieved = row[:6]
    assert reporting_count == len(reporters[code])
    if reached is not None:
        assert reached >= 0, f'Unexpected negative reached total: {code}'
    if achieved is not None:
        assert achieved >= 0, f'Unexpected negative achieved total: {code}'

report_workbook.save(OUTPUT_FILE)
report_workbook.close()
print(f'Output saved: {OUTPUT_FILE.resolve()}')
print('Output sheets: PSR, Source Breakdown, Reference Comparison')
for indicator in ('ED_MS_062', 'ED_MS_066'):
    row = next((r for r in comparison_rows if r[0] == indicator), None)
    if row:
        print(f'{indicator}: reached={row[2]}, achieved={row[5]}')
print('Note: Excel will evaluate formulas when the generated workbook is opened.')

Output saved: /Users/ajgrover/Documents/Scripting/liberia-annual-reporting-26/Liberia_PSR_Bong_Nimba_FY26_Aggregated.xlsx
Output sheets: PSR, Source Breakdown, Reference Comparison
ED_MS_062: reached=401, achieved=None
ED_MS_066: reached=399, achieved=230
Note: Excel will evaluate formulas when the generated workbook is opened.
